# Speaker Verification using ResNet34

Identical to the ResNet18 baseline except the backbone is upgraded to **ResNet34**, which has deeper residual layers for potentially richer speaker embeddings.

---

## 1. Dataset Setup

In [ ]:
import os
import pandas as pd
import numpy as np

DATASET_ROOT = "/kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset"
BASE_AUDIO_DIR = os.path.join(DATASET_ROOT, "train-clean-100", "train-clean-100")
CSV_PATH = os.path.join(DATASET_ROOT, "train_pairs.csv")

print("BASE_AUDIO_DIR:", BASE_AUDIO_DIR)
print("CSV_PATH:", CSV_PATH)

## 2. Load and Inspect Training Data

Load the training CSV and confirm label balance.

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Total rows:", len(df))
print("Label distribution:")
print(df["label"].value_counts())

## 3. Resolve Absolute Audio Paths

Strip the `train-clean-100/` prefix and join with the true dataset root.

In [ ]:
def to_absolute_path(rel_path):
    cleaned = rel_path.replace("train-clean-100/", "", 1)
    return os.path.join(BASE_AUDIO_DIR, cleaned)

df["path1_abs"] = df["audio_path_1"].apply(to_absolute_path)
df["path2_abs"] = df["audio_path_2"].apply(to_absolute_path)
print("Sample path:", df["path1_abs"].iloc[0])

## 4. Audio Transforms & Standardization

Center crop or zero-pad each file to exactly 3 seconds, then extract Log-Mel Spectrograms.

In [ ]:
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T

TARGET_SR = 16000
TARGET_LENGTH = TARGET_SR * 3  # 48000 samples

def crop_or_pad(audio):
    length = len(audio)
    if length > TARGET_LENGTH:
        start = np.random.randint(0, length - TARGET_LENGTH)
        audio = audio[start:start + TARGET_LENGTH]
    elif length < TARGET_LENGTH:
        audio = np.pad(audio, (0, TARGET_LENGTH - length), mode='constant')
    return audio

mel_transform = T.MelSpectrogram(
    sample_rate=16000, n_fft=400, hop_length=160, n_mels=80
)
amplitude_to_db = T.AmplitudeToDB()

## 5. Custom Dataset Class

In [ ]:
import soundfile as sf
from torch.utils.data import Dataset

class SpeakerPairDataset(Dataset):
    def __init__(self, dataframe, mel_transform, amplitude_to_db):
        self.df = dataframe
        self.mel_transform = mel_transform
        self.amplitude_to_db = amplitude_to_db

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        audio, sr = sf.read(path)
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        audio = crop_or_pad(audio)
        audio = torch.tensor(audio).float().unsqueeze(0)
        mel = self.mel_transform(audio)
        return self.amplitude_to_db(mel)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mel1 = self.load_audio(row["path1_abs"])
        mel2 = self.load_audio(row["path2_abs"])
        label = torch.tensor(row["label"]).float()
        return mel1, mel2, label

## 6. Initialize Training Dataset & DataLoader

In [ ]:
from torch.utils.data import DataLoader

dataset = SpeakerPairDataset(df, mel_transform, amplitude_to_db)
print("Dataset size:", len(dataset))

dataloader = DataLoader(
    dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True
)
print("DataLoader ready")

## 7. Model Architecture — ResNetSpeaker (ResNet34)

Only change from the ResNet18 version: `resnet18()` → `resnet34()`.
ResNet34 has 34 layers (vs 18), giving it deeper feature extraction at the cost of slightly more compute.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class ResNetSpeaker(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()

        # *** ResNet34 instead of ResNet18 ***
        self.backbone = models.resnet34(weights=None)

        # Modify first conv to accept 1-channel Log-Mel input
        self.backbone.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.embedding = nn.Linear(in_features, embedding_dim)

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embedding(features)
        return F.normalize(embedding, p=2, dim=1)

## 8. Initialize Model on Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNetSpeaker(embedding_dim=256).to(device)
print("Using device:", device)
print(model)

## 9. Contrastive Loss Function

In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        distance = F.pairwise_distance(emb1, emb2)
        pos_loss = label * torch.pow(distance, 2)
        neg_loss = (1 - label) * torch.pow(torch.clamp(self.margin - distance, min=0.0), 2)
        return torch.mean(pos_loss + neg_loss)

## 10. Training Configuration

Reinitialize the model (clean weights), define the loss and Adam optimizer.

In [ ]:
model = ResNetSpeaker(256).to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 11. Training Loop

Train for **30 epochs** using contrastive loss. Live loss shown via `tqdm`.

In [ ]:
from tqdm import tqdm

num_epochs = 30
print_every = 50

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    progress_bar = tqdm(enumerate(dataloader),
                        total=len(dataloader),
                        desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, (mel1, mel2, labels) in progress_bar:
        mel1 = mel1.to(device)
        mel2 = mel2.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        emb1 = model(mel1)
        emb2 = model(mel2)
        loss = criterion(emb1, emb2, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

        if batch_idx % print_every == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    print(f"\nEpoch [{epoch+1}/{num_epochs}] - Avg Loss: {avg_loss:.4f}\n")

## 12. Save the Trained Model

Save weights, epoch, and optimizer state to `checkpoint_resnet34.pth`.

In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "epoch": epoch,
    "optimizer_state": optimizer.state_dict()
}, "checkpoint_resnet34.pth")

print("Model saved to checkpoint_resnet34.pth")